In [ ]:
!pip install catboost scikit-learn -q

In [ ]:
print('Импорт основных библиотек...')
import pandas as pd
import numpy as np
import io
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
TRAINING_DOWNLOAD_LINK = '10_msh5-vepg-YuB1EmwZZFV7W7D0m8iT'
TESTING_DOWNLOAD_LINK = '1-S2XD6FEoGFNp4c2YYC8TRgvONcbJAIq'

print('Скачивание тренировочных данных...')
!wget -O training_data.csv $TRAINING_DOWNLOAD_LINK

print('Скачивание тестовых данных...')
!wget -O testing_data.csv $TESTING_DOWNLOAD_LINK

training = pd.read_csv('training_data.csv', index_col=0)
testing = pd.read_csv('testing_data.csv', index_col=0)

print('\n')

print('Размер тренировочной выборки:', training.shape)
print('Размер тестовой выборки:', testing.shape)

In [ ]:
all_data = pd.concat([training.drop(columns=['user_id']), testing], axis=0, ignore_index=False)
all_data['time'] = pd.to_datetime(all_data['time'])

all_data['hour'] = all_data['time'].dt.hour
all_data['week'] = all_data['time'].dt.dayofweek
all_data['month'] = all_data['time'].dt.month

bins = [0, 6, 10, 17, 21, 24]
labels = ['night', 'early_morning', 'day', 'evening', 'late_evening']
all_data['time_of_day'] = pd.cut(all_data['hour'], labels=labels, right=False, ordered=False).astype(object)
all_data['is_weekend'] = all_data['week'].isin([5, 6]).astype(int)

training_data_agg = all_data[all_data['user_word'].isnull()].copy()
training_data_agg['user_id'] = training['user_id']

def get_most_frequent(df, group_col, target_col):
    return df.groupby([group_col, target_col]).size().reset_index(name='count').loc[lambda x: x.groupby(group_col)['count'].idxmax()].set_index(group_col)[target_col].to_dict()

favorite_gate_map = get_most_frequent(training_data_agg, 'user_id', 'gate_id')
favorite_hour_map = get_most_frequent(training_data_agg, 'user_id', 'hour')
favorite_week_map = get_most_frequent(training_data_agg, 'user_id', 'week')

all_data['is_favorite_gate'] = (all_data['gate_id'] == all_data['user_id'].map(favorite_gate_map).fillna(-1)).astype(int)
all_data['is_favorite_hour'] = (all_data['hour'] == all_data['user_id'].map(favorite_hour_map).fillna(-1)).astype(int)
all_data['is_favorite_week'] = (all_data['week'] == all_data['user_id'].map(favorite_week_map).fillna(-1)).astype(int)

gate_hour_counts = all_data.groupby(['gate_id', 'hour']).size().to_dict()
all_data['gate_hour_frequent'] = all_data.apply(lambda row: gate_hour_counts.get((row['gate_id'], row['hour']), 0), axis=1)

gate_dow_counts = all_data.groupby(['gate_id', 'week']).size().to_dict()
all_data['gate_week_frequent'] = all_data.apply(lambda row: gate_dow_counts.get((row['gate_id'], row['week']), 0), axis=1)

categorical_features = ['gate_id', 'week', 'month', 'time_of_day']
all_data = pd.get_dummies(all_data, columns=categorical_features, drop_first=True)

all_features = [column for column in all_data.columns if column not in ['time', 'user_word']]

training_final = all_data[all_data['user_word'].isnull()].copy()
testing_final = all_data[all_data['user_word'].notnull()].copy()
training_final['user_id'] = training['user_id']

features_full = training_final[all_features].astype(float)
features_testing_full = testing_final[all_features].astype(float)
target = training_final['user_id'].astype(int)
CLASSES_NUMBER = target.nunique()

print(f'Итоговое количество признаков: {len(all_features)}')
print(f'Количество уникальных пользователей (классов): {CLASSES_NUMBER}')

In [ ]:
features_l_1, features_l_2, target_l_1, target_l_2 = train_test_split(features_full, target, test_size=0.3, random_state=42, stratify=target)

time_gate_features = [column for column in features_full.columns if 'gate_id' in column or 'hour' in column or 'dayofweek' in column or 'time_of_day' in column]

agg_frequent_features = [column for column in features_full.columns if 'is_favorite' in column or 'frequent' in column]

print('Обучение трех базовых моделей...')

model_1 = CatBoostClassifier(
    iterations=700,
    learning_rate=0.05,
    depth=7,
    loss_function='MultiClass',
    random_seed=42,
    verbose=0,
    allow_writing_files=False
)
model_1.fit(features_l_1, target_l_1)

model_2 = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    loss_function='MultiClass',
    random_seed=42,
    verbose=0,
    allow_writing_files=False
)
model_2.fit(features_l_1[time_gate_features], target_l_1)

model_3 = CatBoostClassifier(
    iterations=900,
    learning_rate=0.04,
    depth=9,
    loss_function='MultiClass',
    random_seed=42,
    verbose=0,
    allow_writing_files=False
)
model_3.fit(features_l_1[agg_frequent_features], target_l_1)

In [ ]:
print('Создание признаков для мета-модели...')

probability_model_1 = model_1.predict_proba(features_l_2)
probability_model_2 = model_2.predict_proba(features_l_2[time_gate_features])
probability_model_3 = model_3.predict_proba(features_l_2[agg_frequent_features])

features_meta = np.hstack([probability_model_1, probability_model_2, probability_model_3])

print(f'Размер features_meta: {features_meta.shape}. (3 x {CLASSES_NUMBER} признаков)')

meta_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=4,
    loss_function='MultiClass',
    random_seed=42,
    verbose=0,
    allow_writing_files=False
)

print('Обучение мета-модели...')
meta_model.fit(features_meta, target_l_2)

In [ ]:
print('Применение стекинга к тестовым данным...')

probability_model_1_testing = model_1.predict_proba(features_testing_full)
probability_model_2_testing = model_2.predict_proba(features_testing_full[time_gate_features])
probability_model_3_testing = model_3.predict_proba(features_testing_full[agg_frequent_features])

features_meta_testing = np.hstack([probability_model_1_testing, probability_model_2_testing, probability_model_3_testing])

final_probability = meta_model.predict_proba(features_meta_testing)

final_predicted_id = meta_model.predict(features_meta_testing).flatten()
maximum_probability = np.max(final_probability, axis=1)

testing_words = pd.DataFrame()
testing_words['user_word'] = testing_final['user_word']
testing_words['preds'] = final_predicted_id
testing_words['max_proba'] = maximum_probability

THRESHOLD = 0.99

testing_words.loc[testing_words['max_proba'] < THRESHOLD, 'preds'] = -999

submission = pd.DataFrame(testing_words.groupby('user_word')['preds'].agg(lambda x: x.value_counts().index[0]))

submission.columns = ['preds']
submission['preds'] = submission['preds'].astype(int)

print('\n')

print('Финальные предсказания (полученные с помощью стекинга):')
print(submission)

In [ ]:
submission.to_csv('submission.csv')